In [3]:
# ============================================================
# VideoMAE — Full Pipeline: Train + Evaluasi + Confusion Matrix
# Diadaptasi dari kode kating untuk HPC code-server (bengio)
# Hapus semua Google Colab dependency
# ============================================================

# Jalankan dulu di terminal sebelum run script ini:
# pip install pytorchvideo transformers evaluate scikit-learn imageio av seaborn matplotlib

import sys, os, types, json, shutil
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, tidak butuh display
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Fix torchvision compatibility
from torchvision.transforms.functional import rgb_to_grayscale
functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
functional_tensor.rgb_to_grayscale = rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

from pytorchvideo.data import LabeledVideoDataset, make_clip_sampler
from pytorchvideo.transforms import (
    ApplyTransformToKey, Normalize,
    RandomShortSideScale, UniformTemporalSubsample,
)
from torchvision.transforms import (
    Compose, Lambda, RandomCrop, RandomHorizontalFlip, Resize,
)
from transformers import (
    VideoMAEImageProcessor, VideoMAEForVideoClassification,
    TrainingArguments, Trainer,
)
import evaluate
import pytorchvideo.data

# ============================================================
# PATH CONFIG — ubah di sini saja
# ============================================================
DATA_DIR     = "/home/coder/data_skripsi/dataset_teh_mutia"
OUTPUT_DIR   = "/home/coder/output_model/videomae_5e-5_teh_mutia"
LOG_DIR      = f"{OUTPUT_DIR}/logs"
CORRECT_DIR  = f"{OUTPUT_DIR}/classified_videos/correct"
INCORRECT_DIR= f"{OUTPUT_DIR}/classified_videos/incorrect"

# Pakai pretrained HuggingFace — tidak butuh checkpoint kating
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

for d in [OUTPUT_DIR, LOG_DIR, CORRECT_DIR, INCORRECT_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================
def create_df_from_split(split):
    split_dir = os.path.join(DATA_DIR, split)
    paths, labels = [], []
    for label_name in sorted(os.listdir(split_dir)):
        label_dir = os.path.join(split_dir, label_name)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith((".mp4", ".avi")):
                paths.append(os.path.join(label_dir, fname))
                labels.append(label_name)
    return pd.DataFrame({"Path": paths, "Label": labels})

train_df = create_df_from_split("train")
val_df   = create_df_from_split("val")
test_df  = create_df_from_split("test")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

class_labels = sorted(train_df['Label'].unique())
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}
print("label2id:", label2id)

# ============================================================
# LOAD MODEL
# ============================================================
image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(device)

# ============================================================
# TRANSFORM CONFIG
# ============================================================
mean = image_processor.image_mean
std  = image_processor.image_std

if "shortest_edge" in image_processor.size:
    height = width = image_processor.size["shortest_edge"]
else:
    height = image_processor.size["height"]
    width  = image_processor.size["width"]
resize_to = (height, width)

num_frames_to_sample = model.config.num_frames
sample_rate   = 4
fps           = 30
clip_duration = num_frames_to_sample * sample_rate / fps

print(f"Frames: {num_frames_to_sample} | Clip: {clip_duration:.2f}s | Resize: {resize_to}")

train_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        RandomShortSideScale(min_size=256, max_size=320),
        RandomCrop(resize_to),
        RandomHorizontalFlip(p=0.5),
    ])),
])

val_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        Resize(resize_to),
    ])),
])

# ============================================================
# DATASET
# ============================================================
class CustomVideoDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        return row['Path'], label2id[row['Label']]

def make_labeled_paths(df):
    ds = CustomVideoDataset(df)
    return [(p, {'label': l}) for p, l in ds]

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(train_df),
    clip_sampler=make_clip_sampler("random", clip_duration),
    decode_audio=False, transform=train_transform,
)
val_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(val_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False, transform=val_transform,
)
test_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(test_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False, transform=val_transform,
)

# ============================================================
# METRICS & COLLATE
# ============================================================
accuracy_metric  = evaluate.load("accuracy")
f1_metric        = evaluate.load("f1")
recall_metric    = evaluate.load("recall")
precision_metric = evaluate.load("precision")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    refs  = eval_pred.label_ids
    return {
        "accuracy" : accuracy_metric.compute( predictions=preds, references=refs)["accuracy"],
        "f1"       : f1_metric.compute(       predictions=preds, references=refs, average="weighted")["f1"],
        "recall"   : recall_metric.compute(   predictions=preds, references=refs, average="weighted")["recall"],
        "precision": precision_metric.compute(predictions=preds, references=refs, average="weighted")["precision"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([e["video"].permute(1, 0, 2, 3) for e in examples])
    labels       = torch.tensor([e["label"] for e in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# ============================================================
# TRAINING
# ============================================================
num_epochs  = 20
batch_size  = 4
lr          = 5e-5   # sama persis dengan kating
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir=LOG_DIR,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    max_steps=(len(train_df) // batch_size) * num_epochs,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("\n=== Mulai Training ===")
train_results = trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
with open(os.path.join(OUTPUT_DIR, "train_results.json"), "w") as f:
    json.dump(train_results.metrics, f, indent=2)
print("Training selesai.")

# ============================================================
# EVALUASI — load best model dari final_model
# ============================================================
print("\n=== Load Best Model untuk Evaluasi ===")
final_model_path = os.path.join(OUTPUT_DIR, "final_model")
image_processor  = VideoMAEImageProcessor.from_pretrained(final_model_path)
model = VideoMAEForVideoClassification.from_pretrained(
    final_model_path,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(device)

# Re-init trainer dengan model terbaru
eval_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    per_device_eval_batch_size=batch_size,
    report_to="none",
)

eval_trainer = Trainer(
    model=model,
    args=eval_args,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# Val set
print("\n--- Evaluasi Val Set ---")
val_results = eval_trainer.evaluate(val_dataset)
print(val_results)

# Test set
print("\n--- Evaluasi Test Set ---")
test_results = eval_trainer.evaluate(test_dataset)
print(test_results)

with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(test_results, f, indent=2)

# ============================================================
# PREDICTIONS + CONFUSION MATRIX
# ============================================================
print("\n=== Generating Predictions & Confusion Matrix ===")
predictions    = eval_trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels      = predictions.label_ids

# Confusion matrix — simpan sebagai PNG (tidak ada display di HPC)
cm     = confusion_matrix(true_labels, predicted_labels)
cm_df  = pd.DataFrame(cm, index=class_labels, columns=class_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — VideoMAE (dataset kating)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.close()
print(f"Confusion matrix disimpan: {cm_path}")

# Classification report
report = classification_report(
    true_labels, predicted_labels,
    target_names=class_labels
)
print("\nClassification Report:\n", report)
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

# ============================================================
# SIMPAN VIDEO BENAR / SALAH (sama seperti kode kating)
# ============================================================
print("\n=== Menyimpan Contoh Video Prediksi ===")
for i in range(min(len(test_df), len(predicted_labels))):
    video_path        = test_df.iloc[i]['Path']
    true_label_str    = id2label[true_labels[i]]
    pred_label_str    = id2label[predicted_labels[i]]
    video_filename    = os.path.basename(video_path)

    if true_labels[i] == predicted_labels[i]:
        save_dir = os.path.join(CORRECT_DIR, true_label_str)
    else:
        save_dir = os.path.join(INCORRECT_DIR,
                                f"True_{true_label_str}_Pred_{pred_label_str}")

    os.makedirs(save_dir, exist_ok=True)
    dst = os.path.join(save_dir, video_filename)
    try:
        shutil.copyfile(video_path, dst)
    except Exception as e:
        print(f"  Gagal copy {video_filename}: {e}")

# Simpan predictions ke CSV
n = min(len(test_df), len(predicted_labels), len(true_labels))
results_df = pd.DataFrame({
    'Video Path'     : test_df['Path'].tolist()[:n],
    'True Label'     : [id2label[i] for i in true_labels[:n]],
    'Predicted Label': [id2label[i] for i in predicted_labels[:n]],
    'Correct'        : (true_labels[:n] == predicted_labels[:n]),
})
csv_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
results_df.to_csv(csv_path, index=False)
print(f"Predictions CSV: {csv_path}")

print("\n=== Selesai ===")
print(f"Output tersimpan di: {OUTPUT_DIR}")

Device: cuda
Train: 321 | Val: 40 | Test: 42
label2id: {'1_mengangguk': 0, '2_mengangkat_tangan': 1, '3_menggunakan_hp': 2, '4_menopang_kepala': 3, '5_menunduk': 4}


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 10333.50it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.head.bias                                                    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECT

Frames: 16 | Clip: 2.13s | Resize: (224, 224)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



=== Mulai Training ===


...


--- Evaluasi Val Set ---


Training Loss,Validation Loss,Step,Accuracy,F1,Recall,Precision
No log,0.963588,0,0.738636,0.740911,0.738636,0.766987


{'eval_loss': 0.963588297367096, 'eval_accuracy': 0.7386363636363636, 'eval_f1': 0.7409110653548918, 'eval_recall': 0.7386363636363636, 'eval_precision': 0.7669871994034675}

--- Evaluasi Test Set ---


Training Loss,Validation Loss,Step,Accuracy,F1,Recall,Precision
No log,1.181676,0,0.659574,0.656794,0.659574,0.720937


{'eval_loss': 1.1816760301589966, 'eval_accuracy': 0.6595744680851063, 'eval_f1': 0.6567944250871081, 'eval_recall': 0.6595744680851063, 'eval_precision': 0.7209365020115299}

=== Generating Predictions & Confusion Matrix ===
Confusion matrix disimpan: /home/coder/output_model/videomae_5e-5_teh_mutia/confusion_matrix.png

Classification Report:
                      precision    recall  f1-score   support

       1_mengangguk       1.00      0.33      0.50        15
2_mengangkat_tangan       0.79      0.94      0.86        16
   3_menggunakan_hp       0.74      0.64      0.68        22
  4_menopang_kepala       0.71      0.71      0.71        24
         5_menunduk       0.41      0.65      0.50        17

           accuracy                           0.66        94
          macro avg       0.73      0.65      0.65        94
       weighted avg       0.72      0.66      0.66        94


=== Menyimpan Contoh Video Prediksi ===
Predictions CSV: /home/coder/output_model/videomae_5e-5_teh

In [4]:
# ============================================================
# VideoMAE — Full Pipeline: Train + Evaluasi + Confusion Matrix
# Diadaptasi dari kode kating untuk HPC code-server (bengio)
# Hapus semua Google Colab dependency
# ============================================================

# Jalankan dulu di terminal sebelum run script ini:
# pip install pytorchvideo transformers evaluate scikit-learn imageio av seaborn matplotlib

import sys, os, types, json, shutil
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # non-interactive backend, tidak butuh display
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Fix torchvision compatibility
from torchvision.transforms.functional import rgb_to_grayscale
functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
functional_tensor.rgb_to_grayscale = rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

from pytorchvideo.data import LabeledVideoDataset, make_clip_sampler
from pytorchvideo.transforms import (
    ApplyTransformToKey, Normalize,
    RandomShortSideScale, UniformTemporalSubsample,
)
from torchvision.transforms import (
    Compose, Lambda, RandomCrop, RandomHorizontalFlip, Resize,
)
from transformers import (
    VideoMAEImageProcessor, VideoMAEForVideoClassification,
    TrainingArguments, Trainer,
)
import evaluate
import pytorchvideo.data

# ============================================================
# PATH CONFIG — ubah di sini saja
# ============================================================
DATA_DIR     = "/home/coder/data_skripsi/dataset_raffi"
OUTPUT_DIR   = "/home/coder/output_model/videomae_5e-5_raffi"
LOG_DIR      = f"{OUTPUT_DIR}/logs"
CORRECT_DIR  = f"{OUTPUT_DIR}/classified_videos/correct"
INCORRECT_DIR= f"{OUTPUT_DIR}/classified_videos/incorrect"

# Pakai pretrained HuggingFace — tidak butuh checkpoint kating
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

for d in [OUTPUT_DIR, LOG_DIR, CORRECT_DIR, INCORRECT_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# LOAD DATASET
# ============================================================
def create_df_from_split(split):
    split_dir = os.path.join(DATA_DIR, split)
    paths, labels = [], []
    for label_name in sorted(os.listdir(split_dir)):
        label_dir = os.path.join(split_dir, label_name)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith((".mp4", ".avi")):
                paths.append(os.path.join(label_dir, fname))
                labels.append(label_name)
    return pd.DataFrame({"Path": paths, "Label": labels})

train_df = create_df_from_split("train")
val_df   = create_df_from_split("val")
test_df  = create_df_from_split("test")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

class_labels = sorted(train_df['Label'].unique())
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}
print("label2id:", label2id)

# ============================================================
# LOAD MODEL
# ============================================================
image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(device)

# ============================================================
# TRANSFORM CONFIG
# ============================================================
mean = image_processor.image_mean
std  = image_processor.image_std

if "shortest_edge" in image_processor.size:
    height = width = image_processor.size["shortest_edge"]
else:
    height = image_processor.size["height"]
    width  = image_processor.size["width"]
resize_to = (height, width)

num_frames_to_sample = model.config.num_frames
sample_rate   = 4
fps           = 30
clip_duration = num_frames_to_sample * sample_rate / fps

print(f"Frames: {num_frames_to_sample} | Clip: {clip_duration:.2f}s | Resize: {resize_to}")

train_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        RandomShortSideScale(min_size=256, max_size=320),
        RandomCrop(resize_to),
        RandomHorizontalFlip(p=0.5),
    ])),
])

val_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        Resize(resize_to),
    ])),
])

# ============================================================
# DATASET
# ============================================================
class CustomVideoDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        return row['Path'], label2id[row['Label']]

def make_labeled_paths(df):
    ds = CustomVideoDataset(df)
    return [(p, {'label': l}) for p, l in ds]

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(train_df),
    clip_sampler=make_clip_sampler("random", clip_duration),
    decode_audio=False, transform=train_transform,
)
val_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(val_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False, transform=val_transform,
)
test_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(test_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False, transform=val_transform,
)

# ============================================================
# METRICS & COLLATE
# ============================================================
accuracy_metric  = evaluate.load("accuracy")
f1_metric        = evaluate.load("f1")
recall_metric    = evaluate.load("recall")
precision_metric = evaluate.load("precision")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    refs  = eval_pred.label_ids
    return {
        "accuracy" : accuracy_metric.compute( predictions=preds, references=refs)["accuracy"],
        "f1"       : f1_metric.compute(       predictions=preds, references=refs, average="weighted")["f1"],
        "recall"   : recall_metric.compute(   predictions=preds, references=refs, average="weighted")["recall"],
        "precision": precision_metric.compute(predictions=preds, references=refs, average="weighted")["precision"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([e["video"].permute(1, 0, 2, 3) for e in examples])
    labels       = torch.tensor([e["label"] for e in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# ============================================================
# TRAINING
# ============================================================
num_epochs  = 20
batch_size  = 4
lr          = 5e-5   # sama persis dengan kating
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir=LOG_DIR,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    max_steps=(len(train_df) // batch_size) * num_epochs,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("\n=== Mulai Training ===")
train_results = trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
with open(os.path.join(OUTPUT_DIR, "train_results.json"), "w") as f:
    json.dump(train_results.metrics, f, indent=2)
print("Training selesai.")

# ============================================================
# EVALUASI — load best model dari final_model
# ============================================================
print("\n=== Load Best Model untuk Evaluasi ===")
final_model_path = os.path.join(OUTPUT_DIR, "final_model")
image_processor  = VideoMAEImageProcessor.from_pretrained(final_model_path)
model = VideoMAEForVideoClassification.from_pretrained(
    final_model_path,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to(device)

# Re-init trainer dengan model terbaru
eval_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    per_device_eval_batch_size=batch_size,
    report_to="none",
)

eval_trainer = Trainer(
    model=model,
    args=eval_args,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# Val set
print("\n--- Evaluasi Val Set ---")
val_results = eval_trainer.evaluate(val_dataset)
print(val_results)

# Test set
print("\n--- Evaluasi Test Set ---")
test_results = eval_trainer.evaluate(test_dataset)
print(test_results)

with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(test_results, f, indent=2)

# ============================================================
# PREDICTIONS + CONFUSION MATRIX
# ============================================================
print("\n=== Generating Predictions & Confusion Matrix ===")
predictions    = eval_trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels      = predictions.label_ids

# Confusion matrix — simpan sebagai PNG (tidak ada display di HPC)
cm     = confusion_matrix(true_labels, predicted_labels)
cm_df  = pd.DataFrame(cm, index=class_labels, columns=class_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — VideoMAE (dataset kating)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.close()
print(f"Confusion matrix disimpan: {cm_path}")

# Classification report
report = classification_report(
    true_labels, predicted_labels,
    target_names=class_labels
)
print("\nClassification Report:\n", report)
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

# ============================================================
# SIMPAN VIDEO BENAR / SALAH (sama seperti kode kating)
# ============================================================
print("\n=== Menyimpan Contoh Video Prediksi ===")
for i in range(min(len(test_df), len(predicted_labels))):
    video_path        = test_df.iloc[i]['Path']
    true_label_str    = id2label[true_labels[i]]
    pred_label_str    = id2label[predicted_labels[i]]
    video_filename    = os.path.basename(video_path)

    if true_labels[i] == predicted_labels[i]:
        save_dir = os.path.join(CORRECT_DIR, true_label_str)
    else:
        save_dir = os.path.join(INCORRECT_DIR,
                                f"True_{true_label_str}_Pred_{pred_label_str}")

    os.makedirs(save_dir, exist_ok=True)
    dst = os.path.join(save_dir, video_filename)
    try:
        shutil.copyfile(video_path, dst)
    except Exception as e:
        print(f"  Gagal copy {video_filename}: {e}")

# Simpan predictions ke CSV
n = min(len(test_df), len(predicted_labels), len(true_labels))
results_df = pd.DataFrame({
    'Video Path'     : test_df['Path'].tolist()[:n],
    'True Label'     : [id2label[i] for i in true_labels[:n]],
    'Predicted Label': [id2label[i] for i in predicted_labels[:n]],
    'Correct'        : (true_labels[:n] == predicted_labels[:n]),
})
csv_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
results_df.to_csv(csv_path, index=False)
print(f"Predictions CSV: {csv_path}")

print("\n=== Selesai ===")
print(f"Output tersimpan di: {OUTPUT_DIR}")

Device: cuda
Train: 1404 | Val: 172 | Test: 181
label2id: {'1_mengangguk': 0, '2_mengangkat_tangan': 1, '3_menggunakan_hp': 2, '4_menopang_kepala': 3, '5_menunduk': 4}


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 15322.01it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.bias             | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.head.bias                                                    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECT

Frames: 16 | Clip: 2.13s | Resize: (224, 224)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



=== Mulai Training ===


...


--- Evaluasi Val Set ---


Training Loss,Validation Loss,Step,Accuracy,F1,Recall,Precision
No log,0.540474,0,0.914729,0.914679,0.914729,0.916382


{'eval_loss': 0.5404735803604126, 'eval_accuracy': 0.9147286821705426, 'eval_f1': 0.9146791197383933, 'eval_recall': 0.9147286821705426, 'eval_precision': 0.9163817149729333}

--- Evaluasi Test Set ---


Training Loss,Validation Loss,Step,Accuracy,F1,Recall,Precision
No log,0.661858,0,0.888312,0.888617,0.888312,0.895168


{'eval_loss': 0.6618580222129822, 'eval_accuracy': 0.8883116883116883, 'eval_f1': 0.8886174586105592, 'eval_recall': 0.8883116883116883, 'eval_precision': 0.8951683862393229}

=== Generating Predictions & Confusion Matrix ===
Confusion matrix disimpan: /home/coder/output_model/videomae_5e-5_raffi/confusion_matrix.png

Classification Report:
                      precision    recall  f1-score   support

       1_mengangguk       1.00      0.88      0.94        69
2_mengangkat_tangan       0.80      0.99      0.88        75
   3_menggunakan_hp       0.95      0.87      0.91        79
  4_menopang_kepala       0.90      0.91      0.91        89
         5_menunduk       0.84      0.78      0.81        73

           accuracy                           0.89       385
          macro avg       0.90      0.89      0.89       385
       weighted avg       0.90      0.89      0.89       385


=== Menyimpan Contoh Video Prediksi ===
Predictions CSV: /home/coder/output_model/videomae_5e-5_raffi/t